In [3]:
import geopandas as gpd
import pandas as pd 
import numpy as np
from pathlib import Path
from shapely import wkt

In [57]:
rc_lookups = pd.read_csv("output/rc_lookups.csv")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")

In [58]:
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)
rc

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,LDC_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((440445.7388613202 367018.09139...,2.0,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,MULTIPOLYGON (((439116.7765939621 363407.88751...,131.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,MULTIPOLYGON (((194309.88097327633 76200.71289...,10.0,Cornwall,Cornwall,South West,E63013978
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((439008.4176877096 364010.85388...,17.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,MULTIPOLYGON (((438294.8993448653 371476.61951...,8.0,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,MULTIPOLYGON (((111614.72348488995 512113.4056...,59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,MULTIPOLYGON (((578163.6762067255 233984.52018...,14.0,Braintree,Essex,East of England,E63011054
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,MULTIPOLYGON (((401222.3074132583 90489.386812...,46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,MULTIPOLYGON (((255356.56308415128 666668.6817...,50.0,Glasgow City,City of glasgow,Scotland,S45001180


# apply_hierarchical_classification

In [219]:
def apply_hierarchical_classification(df: pd.DataFrame, retail_park_rc_ids=None, id_col="RC_ID") -> pd.DataFrame:
  
    df = df.copy()
    retail_park_rc_ids = set() if retail_park_rc_ids is None else set(retail_park_rc_ids)

    # 1. HANDLE NA VALUES PROPERLY
    # Instead of filling with "NA" before ranking, we rank first 
    # and then handle the NAs to match the R script logic.
    rank_specs = {
        "RGN_rank": "RegionName",
        "CTY_rank": "CountyName",
        "LAD_rank": "LAD24NM",
        "BUA_rank": "gsscode",
    }
    
    for rank_col, group_col in rank_specs.items():
        # Only rank where the group name is NOT null
        mask = df[group_col].notna()
        df.loc[mask, rank_col] = df[mask].groupby(group_col)["LDC_retailN"].rank(
            method="first", ascending=False
        )
        # If it was NA, give it a massive rank so it's never Rank 1
        df[rank_col] = df[rank_col].fillna(999999)

    df["tr_retailN"] = df["LDC_retailN"]
    df["Classification"] = pd.Series(pd.NA, index=df.index)

    def assign(label: str, mask: pd.Series) -> None:
        df.loc[df["Classification"].isna() & mask, "Classification"] = label

    # TIER 1: Regional (With Manual Overrides)
    assign("Regional Centre", (df["RGN_rank"] == 1))

    # TIER 2: Major Town
    assign("Major Town Centre", (df["CTY_rank"] == 1) & (df["RGN_rank"] != 1) & (df["LDC_retailN"] >= 350))

    # TIER 3: Town Centre
    assign("Town Centre", 
        ((df["CTY_rank"] == 1) & (df["RGN_rank"] != 1)) | 
        ((df["LAD_rank"] == 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1)) | 
        ((df["LDC_retailN"] >= 250) & (df["LDC_retailN"] < 350)) 
    )

    # TIER 4: Market Town
    assign("Market Town", 
        (df["BUA_rank"] == 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150) & (df["tr_retailN"] < 250)
    )

    # TIER 5: District Centre
    assign("District Centre", 
        (df["BUA_rank"] != 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150)
    )

    # TIER 6: Local Centre
    assign("Local Centre", df["tr_retailN"] < 150)

    # REFINEMENT: Local -> Small Local
    df.loc[(df["Classification"] == "Local Centre") & (df["tr_retailN"] < 100), "Classification"] = "Small Local Centre"

    # FALLBACKS
    df.loc[df["Classification"].isna() & df["LDC_retailN"].notna(), "Classification"] = "Market Town"
    df["Classification"] = df["Classification"].fillna("Small Local Centre")

    # RETAIL PARK ADJUSTMENT
    # Refined to include District Centres if they are actually Retail Parks
    retail_park_mask = (
        (df["centre_type"] == "retail_park") 
    ) & (df["Classification"].isin(["Small Local Centre"]))
    # retail_park_mask = (
    #     (df["centre_type"] == "retail_park") | 
    #     df["RC_ID"].isin(retail_park_rc_ids)
    # ) & (df["Classification"].isin(["Small Local Centre", "Local Centre", "District Centre"]))
    
    df.loc[retail_park_mask, "Classification"] = "Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] < 50), "Classification"] = "Small Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] >= 50), "Classification"] = "Large Retail Park"

    return df.sort_values(["RegionName", id_col]).reset_index(drop=True)

In [270]:
df= apply_hierarchical_classification(rc)

In [271]:
df[df['RC_ID'] == 994]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
6846,994,161,45,127.0,32.0,1.0,1.0,['Latin American Restaurant' 'Sporting Goods R...,centre,Swindon,...,Swindon,Wiltshire,South West,E34004828,107.0,10.0,2.0,2.0,124.0,Local Centre


# add manual

In [272]:
def apply_manual_edits(
    df: pd.DataFrame,
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
    manual_hierarchy_csv: str | None = None,
    group_col: str = "group_id",   # will auto-skip if not present
) -> pd.DataFrame:
    df = df.copy()

    # --- build old -> list of new mapping ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = {}
    for old_id, new_id in zip(lookup["RC_ID"], lookup["new_rcid"]):
        old_to_new.setdefault(old_id, []).append(new_id)

    def map_old_to_new_list(old_id: str):
        return old_to_new.get(old_id, [old_id])

    has_group = group_col in df.columns

    manual_classification_old = {
        "RC_EW_2920": "Market Town",
        "RC_SC_332": "Major Town Centre",
        "RC_EW_876": "Major Town Centre",
        "RC_EW_5564": "Major Town Centre",
        "RC_EW_61": "Major Town Centre",
        "RC_EW_3096": "Major Town Centre",
        "RC_EW_3254": "Major Town Centre",
        "RC_EW_3301": "Major Town Centre",
        "RC_EW_3476": "Major Town Centre",
        "RC_EW_3522": "Regional Centre",
        "RC_EW_2751": "Regional Centre",
        "RC_SC_280": "Regional Centre",
    }

    for old_id, cls in manual_classification_old.items():
        new_ids = map_old_to_new_list(old_id)

        if has_group:
            # find groups containing any of those RC_IDs, then apply to whole group
            target_groups = df.loc[df["RC_ID"].isin(new_ids), group_col].unique()
            df.loc[df[group_col].isin(target_groups), "Classification"] = cls
        else:
            # no group_id, apply directly
            df.loc[df["RC_ID"].isin(new_ids), "Classification"] = cls

    # --- Manual hierarchy list (upgrade Town Centre -> Major Town Centre) ---
    if manual_hierarchy_csv:
        manual_path = Path(manual_hierarchy_csv)
        if manual_path.exists():
            rc_manual = pd.read_csv(manual_path)
            rc_manual = rc_manual.loc[
                rc_manual["H_Classification"] == "Major Town Centre", "RC_ID"
            ].dropna()

            new_ids = []
            for old_id in rc_manual.astype(str).unique():
                new_ids.extend(map_old_to_new_list(old_id))

            if has_group:
                target_groups = df.loc[df["RC_ID"].isin(new_ids), group_col].unique()
                mask = df[group_col].isin(target_groups) & (df["Classification"] == "Town Centre")
            else:
                mask = df["RC_ID"].isin(new_ids) & (df["Classification"] == "Town Centre")

            df.loc[mask, "Classification"] = "Major Town Centre"

    return df


In [274]:
df1 = apply_manual_edits(df)
df1[df1['RC_ID'] == 994]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
6846,994,161,45,127.0,32.0,1.0,1.0,['Latin American Restaurant' 'Sporting Goods R...,centre,Swindon,...,Swindon,Wiltshire,South West,E34004828,107.0,10.0,2.0,2.0,124.0,Local Centre


# shopping centres

In [284]:
def apply_shopping_outlet_overrides(
    df: pd.DataFrame,
    rc_uk_path: str = "geo/postworkshopcentres.gpkg",
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
) -> pd.DataFrame:
    """
    Manual outlet locations (spatial intersection + explicit ID overrides).
    rc_uk_path: path to postworkshopcentres.gpkg
    rcid_lookup_csv: path to rcid_lookup_old_to_new.csv
    """
    df = df.copy()

    # --- RC_ID mapping (old -> new) ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = dict(zip(lookup["RC_ID"], lookup["new_rcid"]))

    def map_rcid(old_id: str) -> str:
        return old_to_new.get(old_id, old_id)

    # --- Manual outlet locations ---
    outlets = pd.DataFrame(
        [
            ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
            ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
            ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
            ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
            ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
            ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
            ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
            ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
            ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
            ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
            ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
            ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
            ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
            ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
            ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
            ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
            #("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
            ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
            ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
            ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
            ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
            ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
            ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
            ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
            ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
            ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
            ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
            ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
            ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
            ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
            ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
            ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
            ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
            ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
            ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
            ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
            ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
            ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
            ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
            ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
            ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
            ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
            ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
            ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
            ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
            ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
            ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
        ],
        columns=["SubType", "Name", "Lat", "Lon"],
    )
    outlets["Type"] = "Out of Town Shopping Centres"

    outlets_gdf = gpd.GeoDataFrame(
        outlets,
        geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
        crs="EPSG:4326",
    )

    rc_uk = gpd.read_file(rc_uk_path)
    outlets_gdf = outlets_gdf.to_crs(rc_uk.crs)

    outlets_joined = gpd.sjoin(
        outlets_gdf,
        rc_uk[["RC_ID", "geometry"]],
        how="left",
        predicate="intersects",
    )
    outlets_tag = outlets_joined[["Type", "RC_ID"]].dropna(subset=["RC_ID"])

    manual_outlet_rcids_old = {
        "Ashford Designer Outlet": "RC_EW_1312",
        "Chatham Dockside Outlet Centre": "RC_EW_1529",
        "Doncaster Lakeside Village": "RC_EW_762",
        "Swindon Designer Outlet": "RC_EW_3751",
        "Springfields Spalding Outlet": "RC_EW_955",
    }

    manual_override = outlets_gdf[outlets_gdf["Name"].isin(manual_outlet_rcids_old)].copy()
    manual_override["RC_ID"] = manual_override["Name"].map(manual_outlet_rcids_old).map(map_rcid)
    manual_override = manual_override[["Type", "RC_ID"]].dropna(subset=["RC_ID"])
  
    outlets_tag = pd.concat([outlets_tag, manual_override], ignore_index=True)
    outlets_tag = outlets_tag.drop_duplicates(subset=["RC_ID"])

    # df = df.merge(outlets_tag, on="RC_ID", how="left", sort=False)
    # df.loc[df["Type"].notna(), "Classification"] = df.loc[df["Type"].notna(), "Type"]
    # df = df.drop(columns=["Type"])
    df.loc[df["RC_ID"].isin(outlets_tag["RC_ID"]), "Classification"] = "Out of Town Shopping Centres"


    return df


In [289]:
df2 = apply_shopping_outlet_overrides(df1)
df2.shape

(9622, 24)

In [145]:
def apply_out_of_town_sizing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    out_of_town_mask = df["Classification"].isin(["Out of Town Shopping Centres"])

    df.loc[out_of_town_mask & (df["tr_retailN"] < 50), "Classification"] = "Small Shopping Centre"
    df.loc[out_of_town_mask & (df["tr_retailN"] >= 50), "Classification"] = "Large Shopping Centre"

    return df


In [183]:
df3 = apply_out_of_town_sizing(df2)
df3[df3['RC_ID'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
7501,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,Cardiff,South glamorgan,Wales,W37000384,1.0,1.0,1.0,1.0,1001.0,Regional Centre


In [ ]:
df3['Classification'].isna().sum()

In [ ]:
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df3[cols] = df3[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")

In [ ]:
df3["geometry"] = df3["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

# Now create GeoDataFrame with CRS
gdf = gpd.GeoDataFrame(df3, geometry="geometry", crs="EPSG:27700")  # change CRS if needed
hex_map = {
    "Regional Centre": "#882255",
    "Major Town Centre": "#332288",
    "Town Centre": "#117733",
    "Market Town": "#44AA99",
    "District Centre": "#999933",
    "Local Centre": "#DDCC77",
    "Small Local Centre": "#88CCEE",
    "Large Retail Park": "#CC6677",
    "Small Retail Park": "#E0A1AB",
    "Large Out of Town Shopping Centre": "#AA4499",
    "Small Out of Town Shopping Centre": "#CD83C1",
}

gdf["HEX_COL"] = gdf["Classification"].map(hex_map)

# Export (pick your path)
out_path = "output/rc2025_classified.shp"
gdf.to_file(out_path, driver="ESRI Shapefile")

In [ ]:
geo = df3
geo["geometry"] = geo["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

geo_gdf = gpd.GeoDataFrame(geo, geometry="geometry", crs="EPSG:27700")
geo_gdf.to_file("output/rc2025_classified.gpkg", driver="GPKG")


# Join the split togther and using the 2011 build up area

In [248]:
# rc_geo = gpd.read_file("output/rc2025_classified.gpkg")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")
rc_lookups = pd.read_csv("output/rc_lookups_2011BUA.csv") # 2011 BUA version
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)

rc["geometry"] = rc["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

rc_geo = gpd.GeoDataFrame(
    rc,
    geometry="geometry",
    crs="EPSG:4326"  # set to the correct CRS for your coordinates
)
rc_geo.rename(columns={"LDC_retailN": "tr_retailN"}, inplace=True)

In [ ]:
# split info
rc_split = gpd.read_file("geo/postworkshopcentres_withsplit.gpkg")
rc_split.head()

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,was_split,Original_POI_count,Parent_ID,geometry
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,None,None,False,16,NaN,"MULTIPOLYGON (((440445.739 367018.091, 440420...."
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,None,None,False,131,NaN,"MULTIPOLYGON (((439116.777 363407.888, 439111...."
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,None,None,False,14,NaN,"MULTIPOLYGON (((194309.881 76200.713, 194331.3..."
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,None,None,False,16,NaN,"MULTIPOLYGON (((439008.418 364010.854, 438983...."
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,None,None,False,12,NaN,"MULTIPOLYGON (((438294.899 371476.62, 438289.9..."


In [317]:
df = rc_geo.merge(
    rc_split[["RC_ID", "was_split", "Parent_ID"]],
    on="RC_ID",
    how="left"
)
# 1) build group_id first
is_split = df["was_split"].astype(str).str.lower().isin(["true", "1", "yes"])
df["group_id"] = df["RC_ID"].astype(str)
df.loc[is_split, "group_id"] = "P_" + df.loc[is_split, "Parent_ID"].astype(str)

# 2) build group-level table (one row per group_id)
group_df = (
    df.groupby("group_id", dropna=False)
      .agg(
          LDC_retailN=("tr_retailN", "sum"),
          RegionName=("RegionName", "first"),
          CountyName=("CountyName", "first"),
          LAD24NM=("LAD24NM", "first"),
          gsscode=("gsscode", "first"),
          centre_type=("centre_type", "first"),
      )
      .reset_index()
)

# 3) classify at GROUP level only
group_df = apply_hierarchical_classification(group_df, id_col="group_id")
group_df.rename(columns={"LDC_retailN": "group_LDC_retailN"}, inplace=True)

df = df.merge(
    group_df[["group_id", "group_LDC_retailN", "Classification"]],
    on="group_id",
    how="left"
)

df = apply_manual_edits(df)
df = apply_shopping_outlet_overrides(df)
df = apply_out_of_town_sizing(df)
# df.head()

In [316]:
df["RC_ID"].duplicated().sum()

np.int64(0)

In [323]:
def classify_with_grouping(
    rc_geo: pd.DataFrame,
    rc_split: pd.DataFrame,
    parent_col: str = "Parent_ID",
    split_col: str = "was_split",
) -> pd.DataFrame:
    df = rc_geo.merge(
        rc_split[["RC_ID", split_col, parent_col]],
        on="RC_ID",
        how="left"
    )

    # 1) build group_id
    is_split = df[split_col].astype(str).str.lower().isin(["true", "1", "yes"])
    df["group_id"] = df["RC_ID"].astype(str)
    df.loc[is_split, "group_id"] = "P_" + df.loc[is_split, parent_col].astype(str)

    # 2) build group-level table
    group_df = (
        df.groupby("group_id", dropna=False)
          .agg(
              LDC_retailN=("tr_retailN", "sum"),
              RegionName=("RegionName", "first"),
              CountyName=("CountyName", "first"),
              LAD24NM=("LAD24NM", "first"),
              gsscode=("gsscode", "first"),
              centre_type=("centre_type", "first"),
          )
          .reset_index()
    )

    # 3) classify at GROUP level
    group_df = apply_hierarchical_classification(group_df, id_col="group_id")
    group_df.rename(columns={"LDC_retailN": "group_LDC_retailN"}, inplace=True)

    # 4) merge group results back
    df = df.merge(
        group_df[["group_id", "group_LDC_retailN", "Classification"]],
        on="group_id",
        how="left"
    )

    # 5) apply overrides on row level (propagate by group_id inside functions)
    df = apply_manual_edits(df)
    df = apply_shopping_outlet_overrides(df)
    df = apply_out_of_town_sizing(df)

    return df


In [324]:
df_join = classify_with_grouping(
    rc_geo=rc_geo,
    rc_split= gpd.read_file("geo/postworkshopcentres_withsplit.gpkg"),
    parent_col="Parent_ID", 
    split_col="was_split"
)

In [286]:
# export
#cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
cols = ["tr_retailN", 'group_LDC_retailN']

df_join[cols] = df_join[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")
df_join["geometry"] = df_join["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)
out_gdf = gpd.GeoDataFrame(df_join, geometry="geometry", crs="EPSG:27700")
out_path = "output/rc2025_classified_joined_BUA11_2.gpkg"

#out_gdf.to_file(out_path, driver="ESRI Shapefile")
out_gdf.to_file(out_path, driver="GPKG", layer="rc2025_classified_joined_BUA11")


# dissolve the Built-Up Areas

In [247]:
buildup_latest = gpd.read_file("geo/buildup/buildup.shp")


# checking

In [ ]:

# build outlets DataFrame 
outlets = pd.DataFrame(
    [
        ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
        ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
        ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
        ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
        ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
        ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
        ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
        ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
        ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
        ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
        ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
        ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
        ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
        ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
        ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
        ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
        ("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
        ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
        ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
        ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
        ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
        ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
        ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
        ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
        ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
        ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
        ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
        ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
        ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
        ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
        ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
        ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
        ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
        ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
        ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
        ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
        ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
        ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
        ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
        ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
        ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
        ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
        ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
        ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
        ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
        ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
        ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
    ],
    columns=["SubType", "Name", "Lat", "Lon"],
)
outlets["Type"] = "Out of Town Shopping Centres"

# Make GeoDataFrame
outlets_gdf = gpd.GeoDataFrame(
    outlets,
    geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
    crs="EPSG:4326",
)

# Export
outlets_gdf.to_file("output/outlets.shp", driver="ESRI Shapefile")


In [287]:
new_path = "output/rc2025_classified_joined_2011py.gpkg"
old_path = "output/rc2025_classified_joined_BUA11_2.gpkg"

g1 = gpd.read_file(new_path)  # new
g2 = gpd.read_file(old_path)  # old

# 1) duplicate RC_IDs
print("dup new:", g1["RC_ID"].duplicated().sum())
print("dup old:", g2["RC_ID"].duplicated().sum())

dup_old = g2[g2["RC_ID"].duplicated(keep=False)].sort_values("RC_ID")
dup_new = g1[g1["RC_ID"].duplicated(keep=False)].sort_values("RC_ID")

# 2) align on RC_ID for comparisons
# make RC_ID types consistent
g1["RC_ID"] = g1["RC_ID"].astype(str)
g2["RC_ID"] = g2["RC_ID"].astype(str)

# remove duplicates in old (keep first for compare)
g2_nodup = g2.drop_duplicates(subset=["RC_ID"])

m = g1.merge(
    g2_nodup,
    on="RC_ID",
    how="outer",
    suffixes=("_new", "_old"),
    indicator=True
)


print("only in new:", (m["_merge"]=="left_only").sum())
print("only in old:", (m["_merge"]=="right_only").sum())

# 3) Classification differences
cls_diff = m[(m["_merge"]=="both") & (m["Classification_new"] != m["Classification_old"])]
print("Classification diffs:", len(cls_diff))
cls_diff[["RC_ID","Classification_new","Classification_old"]].head(20)

# 4) Column differences summary (common columns only)
common = set(g1.columns) & set(g2.columns) - {"geometry"}
diff_counts = {}
for c in common:
    a = f"{c}_new"
    b = f"{c}_old"
    if a in m.columns and b in m.columns:
        diff_counts[c] = (m[a] != m[b]) & ~(m[a].isna() & m[b].isna())
print({k:int(v.sum()) for k,v in diff_counts.items() if v.sum()>0})


dup new: 0
dup old: 2
only in new: 0
only in old: 0
Classification diffs: 134
{'gsscode': 9298, 'LAD24NM': 1, 'Classification': 134}


In [282]:
g2[g2['RC_ID'].duplicated(keep=False)].sort_values('RC_ID')

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,was_split,Parent_ID,group_id,group_LDC_retailN,Classification,geometry
742,1968,73,117,59.0,10.0,2.0,2.0,['Tourist Information and Service' 'Sporting G...,centre,South Holland,...,South Holland,Lincolnshire,East Midlands,E34004218,False,NaN,1968,72,Large Shopping Centre,"MULTIPOLYGON (((526226.882 323856.686, 526201...."
743,1968,73,117,59.0,10.0,2.0,2.0,['Tourist Information and Service' 'Sporting G...,centre,South Holland,...,South Holland,Lincolnshire,East Midlands,E34004218,False,NaN,1968,72,Large Shopping Centre,"MULTIPOLYGON (((526226.882 323856.686, 526201...."
5092,994,161,45,127.0,32.0,1.0,1.0,['Latin American Restaurant' 'Sporting Goods R...,centre,Swindon,...,Swindon,Wiltshire,South West,E34004828,False,NaN,994,124,Large Shopping Centre,"MULTIPOLYGON (((414044.815 184831.393, 414019...."
5093,994,161,45,127.0,32.0,1.0,1.0,['Latin American Restaurant' 'Sporting Goods R...,centre,Swindon,...,Swindon,Wiltshire,South West,E34004828,False,NaN,994,124,Large Shopping Centre,"MULTIPOLYGON (((414044.815 184831.393, 414019...."
